# New York City, 2024 → 2025: Did the Neighborhoods That Moved to Trump Move to Mamdani?

Part 6 found that the 2024 rightward shift in the bottom income quartile was carried by Hispanic and Black low-income counties; Parts 8–9 found the leavers were economically left, culturally conservative, price-focused, and did not see much economic difference between the candidates. Eleven months later, New York City nominated and elected a democratic socialist mayor on an explicitly economic platform (rent freeze, city-run groceries, free buses), against the best-funded establishment candidate in the race and with hostile coverage across the cable spectrum.

New York is the one place where the same precincts can be observed voting in both contests. The question is direct: **did the election districts that swung hardest to Trump in 2024 vote for Mamdani in 2025 — in the June Democratic primary and in the November general?** If they did, the 2024 leavers were reachable with an economic message; if Mamdani's coalition was the culturally liberal, college-educated core and the Trump-shifted neighborhoods stayed away or went to Cuomo, the Part 9 reading — that the cultural line held — extends to the case most often cited against it.

**Inputs:** `data/raw/nyc/nyc_ed_panel.csv` from `src/data_collection/nyc_pull.py` — ED-level 2024 presidential and 2025 mayoral general returns (BOE), 2025 primary first choices and Mamdani-vs-Cuomo head-to-head built from the cast vote record, and 2020 Census / ACS 2023 block-based demographics.

**Caveats up front.** This is ecological: an ED that moved to Trump and then to Mamdani need not contain the same people doing both. The primary electorate is registered Democrats only; the general electorate in an off-year mayoral race is smaller and older than a presidential one. ED lines changed slightly between the three contests (nyed_24d / 25b / 25d); EDs that do not match across files are dropped, and "combined" EDs (reported under another ED) are excluded.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.api as sm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
RAW_DIR = Path('../data/raw'); TABLES_DIR = Path('../outputs/tables'); FIGURES_DIR = Path('../outputs/figures')
d = pd.read_csv(RAW_DIR / 'nyc' / 'nyc_ed_panel.csv', index_col='ElectDist')
print(d.shape); print(d.columns.tolist())
# 2020 presidential votes are crosswalked from pre-2022 ED lines to current EDs via 2020 census blocks (population-weighted); see nyc_pull.crosswalk_2020.
d['pres24_2p'] = d.pres24_harris + d.pres24_trump
d['harris_share'] = d.pres24_harris / d.pres24_2p
d['trump_share'] = d.pres24_trump / d.pres24_2p
d['gen25_2p'] = d.gen25_mamdani + d.gen25_cuomo
d['mamdani_gen_share'] = d.gen25_mamdani / d.gen25_total          # of all mayoral votes
d['mamdani_vs_cuomo_gen'] = d.gen25_mamdani / d.gen25_2p
d['sliwa_gen_share'] = d.gen25_sliwa / d.gen25_total
d['prim_h2h_total'] = d.prim25_h2h_mamdani + d.prim25_h2h_cuomo
d['mamdani_h2h_prim'] = d.prim25_h2h_mamdani / d.prim_h2h_total
d['mamdani_first_share'] = d.prim25_first_mamdani / d.prim25_ballots
d['cuomo_first_share'] = d.prim25_first_cuomo / d.prim25_ballots
d['turnout_ratio_25_24'] = d.gen25_total / d.pres24_total
d['pres20_2p'] = d.pres20_biden + d.pres20_trump
d['trump_share_20'] = d.pres20_trump / d.pres20_2p
d['swing_to_trump'] = d.trump_share - d.trump_share_20            # 2020 -> 2024, positive = moved right
ok = (d.pres24_2p >= 50) & (d.gen25_2p >= 50) & (d.pres20_2p >= 50) & (~d.combined_into.fillna(False)) & d.pop2020.gt(0)
e = d[ok].copy()
print('EDs usable (>=50 two-party votes in 2020, 2024, and 2025 generals, not combined, populated):', len(e))
print('Citywide 2020 (crosswalked, usable EDs): Trump %.1f%%; 2024: Trump %.1f%%; swing %+.1f pts' % (100*e.pres20_trump.sum()/e.pres20_2p.sum(), 100*e.pres24_trump.sum()/e.pres24_2p.sum(), 100*(e.pres24_trump.sum()/e.pres24_2p.sum() - e.pres20_trump.sum()/e.pres20_2p.sum())))
print('Citywide (usable EDs): Harris %.1f%%, Trump %.1f%% | Mamdani %.1f%% of mayoral vote, %.1f%% of Mamdani+Cuomo, Sliwa %.1f%% | primary Mamdani-Cuomo head-to-head %.1f%%' % (
    100*e.pres24_harris.sum()/e.pres24_2p.sum(), 100*e.pres24_trump.sum()/e.pres24_2p.sum(), 100*e.gen25_mamdani.sum()/e.gen25_total.sum(),
    100*e.gen25_mamdani.sum()/e.gen25_2p.sum(), 100*e.gen25_sliwa.sum()/e.gen25_total.sum(), 100*e.prim25_h2h_mamdani.sum()/e.prim_h2h_total.sum()))

(4326, 35)
['pres24_total', 'pres24_harris', 'pres24_trump', 'county', 'combined_into', 'gen25_total', 'gen25_mamdani', 'gen25_cuomo', 'gen25_sliwa', 'combined_into_g25', 'prim25_ballots', 'prim25_h2h_mamdani', 'prim25_h2h_cuomo', 'prim25_h2h_neither', 'prim25_first_mamdani', 'prim25_first_cuomo', 'prim25_first_lander', 'prim25_first_adams', 'prim25_first_stringer', 'prim25_first_myrie', 'prim25_first_ramos', 'prim25_first_blake', 'prim25_first_tilson', 'pop2020', 'hispanic_share', 'white_nh_share', 'black_nh_share', 'asian_nh_share', 'med_income', 'ba_share', 'poverty', 'renter', 'pres20_biden', 'pres20_trump', 'pres20_total']
EDs usable (>=50 two-party votes in 2020, 2024, and 2025 generals, not combined, populated): 3870
Citywide 2020 (crosswalked, usable EDs): Trump 23.1%; 2024: Trump 31.0%; swing +7.9 pts
Citywide (usable EDs): Harris 69.0%, Trump 31.0% | Mamdani 51.0% of mayoral vote, 55.3% of Mamdani+Cuomo, Sliwa 7.0% | primary Mamdani-Cuomo head-to-head 56.6%


## 1. The 2020→2024 swing and the 2025 maps

The 2024 shift is measured directly: the change in Trump's two-party share from 2020 (crosswalked to current ED lines through census blocks) to 2024. EDs are sorted into deciles of that swing and of the 2024 Trump level, and Mamdani's June and November 2025 performance is read off each.

In [2]:
def wcorr(x, y, w):
    m = x.notna() & y.notna() & w.notna()
    x, y, w = x[m], y[m], w[m]
    mx, my = np.average(x, weights=w), np.average(y, weights=w)
    return np.average((x-mx)*(y-my), weights=w) / np.sqrt(np.average((x-mx)**2, weights=w) * np.average((y-my)**2, weights=w))
print('ED-level correlations (weighted by 2024 two-party votes):')
print('%-24s %14s %16s' % ('', 'w/ 2024 Trump', 'w/ swing 20->24'))
for c in ['mamdani_gen_share', 'mamdani_vs_cuomo_gen', 'sliwa_gen_share', 'mamdani_h2h_prim', 'mamdani_first_share', 'cuomo_first_share', 'turnout_ratio_25_24']:
    print('%-24s %+14.3f %+16.3f' % (c, wcorr(e.trump_share, e[c], e.pres24_2p), wcorr(e.swing_to_trump, e[c], e.pres24_2p)))
print('corr(swing, 2024 Trump share) = %+.3f' % wcorr(e.swing_to_trump, e.trump_share, e.pres24_2p))
# deciles of the 2020->2024 swing, then of the 2024 level
e['swing_decile'] = pd.qcut(e.swing_to_trump, 10, labels=False) + 1
e['trump_decile'] = pd.qcut(e.trump_share, 10, labels=False) + 1
def decile_table(by):
  return e.groupby(by).apply(lambda s: pd.Series({
    'trump_share_20': s.pres20_trump.sum()/s.pres20_2p.sum(), 'trump_share_24': s.pres24_trump.sum()/s.pres24_2p.sum(), 'swing': s.pres24_trump.sum()/s.pres24_2p.sum() - s.pres20_trump.sum()/s.pres20_2p.sum(), 'mamdani_gen': s.gen25_mamdani.sum()/s.gen25_total.sum(), 'cuomo_gen': s.gen25_cuomo.sum()/s.gen25_total.sum(),
    'sliwa_gen': s.gen25_sliwa.sum()/s.gen25_total.sum(), 'mamdani_vs_cuomo_gen': s.gen25_mamdani.sum()/s.gen25_2p.sum(),
    'mamdani_h2h_prim': s.prim25_h2h_mamdani.sum()/s.prim_h2h_total.sum(), 'mamdani_first': s.prim25_first_mamdani.sum()/s.prim25_ballots.sum(),
    'prim_ballots_per_24_vote': s.prim25_ballots.sum()/s.pres24_total.sum(), 'gen25_per_24_vote': s.gen25_total.sum()/s.pres24_total.sum(),
    'hispanic': np.average(s.hispanic_share, weights=s.pop2020), 'black': np.average(s.black_nh_share, weights=s.pop2020), 'white': np.average(s.white_nh_share, weights=s.pop2020),
    'asian': np.average(s.asian_nh_share, weights=s.pop2020), 'med_income': np.average(s.med_income.fillna(s.med_income.median()), weights=s.pop2020), 'ba_share': np.average(s.ba_share.fillna(0), weights=s.pop2020), 'N': len(s)}))
decs = decile_table('swing_decile'); print('\nBy decile of the 2020->2024 swing to Trump (1 = moved least / left, 10 = moved most right):'); print(decs.round(3))
dec = decile_table('trump_decile'); print('\nBy decile of 2024 Trump share:'); print(dec.round(3))
decs.to_csv(TABLES_DIR / 'nyc_by_swing_decile.csv'); dec.to_csv(TABLES_DIR / 'nyc_by_trump_decile.csv')

ED-level correlations (weighted by 2024 two-party votes):
                          w/ 2024 Trump  w/ swing 20->24
mamdani_gen_share                -0.801           -0.099
mamdani_vs_cuomo_gen             -0.764           -0.067
sliwa_gen_share                  +0.737           +0.137
mamdani_h2h_prim                 -0.313           -0.051
mamdani_first_share              -0.182           +0.105
cuomo_first_share                +0.367           +0.124
turnout_ratio_25_24              +0.106           +0.125
corr(swing, 2024 Trump share) = +0.360

By decile of the 2020->2024 swing to Trump (1 = moved least / left, 10 = moved most right):
              trump_share_20  trump_share_24  swing  mamdani_gen  cuomo_gen  \
swing_decile                                                                  
1                      0.162           0.160 -0.003        0.638      0.323   
2                      0.176           0.202  0.026        0.563      0.383   
3                      0.190          

## 2. The Part 6 neighborhoods: Hispanic, Black, Asian, and low-income EDs

Part 6's 2024 movers were nonwhite low-income places. Here each ED is typed by its plurality group (2020 Census) and income tercile, and the three contests are compared within type.

In [3]:
e['plurality'] = e[['hispanic_share', 'white_nh_share', 'black_nh_share', 'asian_nh_share']].idxmax(axis=1).str.replace('_share', '').str.replace('_nh', '')
e['majority'] = np.where(e[['hispanic_share', 'white_nh_share', 'black_nh_share', 'asian_nh_share']].max(axis=1) >= 0.5, e.plurality, 'mixed')
e['income_tercile'] = pd.qcut(e.med_income, 3, labels=['low', 'mid', 'high'])
def agg(s):
    return pd.Series({'N': len(s), 'pop': s.pop2020.sum(), 'trump_20': s.pres20_trump.sum()/s.pres20_2p.sum(), 'trump_24': s.pres24_trump.sum()/s.pres24_2p.sum(), 'swing': s.pres24_trump.sum()/s.pres24_2p.sum() - s.pres20_trump.sum()/s.pres20_2p.sum(),
        'mamdani_gen': s.gen25_mamdani.sum()/s.gen25_total.sum(), 'cuomo_gen': s.gen25_cuomo.sum()/s.gen25_total.sum(), 'sliwa_gen': s.gen25_sliwa.sum()/s.gen25_total.sum(),
        'mamdani_vs_cuomo_gen': s.gen25_mamdani.sum()/s.gen25_2p.sum(), 'mamdani_h2h_prim': s.prim25_h2h_mamdani.sum()/s.prim_h2h_total.sum(),
        'mamdani_first': s.prim25_first_mamdani.sum()/s.prim25_ballots.sum(), 'cuomo_first': s.prim25_first_cuomo.sum()/s.prim25_ballots.sum(),
        'gen25_per_24_vote': s.gen25_total.sum()/s.pres24_total.sum(), 'prim_per_24_vote': s.prim25_ballots.sum()/s.pres24_total.sum()})
by_maj = e.groupby('majority').apply(agg); print('By majority group:'); print(by_maj.round(3))
by_inc = e.groupby('income_tercile').apply(agg); print('\nBy income tercile:'); print(by_inc.round(3))
cross = e.groupby(['majority', 'income_tercile']).apply(agg); print('\nMajority x income:'); print(cross.round(3))
by_maj.to_csv(TABLES_DIR / 'nyc_by_majority.csv'); cross.to_csv(TABLES_DIR / 'nyc_by_majority_x_income.csv')

By majority group:
               N        pop  trump_20  trump_24  swing  mamdani_gen  \
majority                                                              
asian      226.0   601437.0     0.321     0.469  0.148        0.465   
black      628.0  1356712.0     0.055     0.101  0.045        0.636   
hispanic   723.0  1805994.0     0.162     0.298  0.136        0.584   
mixed     1060.0  2342433.0     0.222     0.307  0.086        0.577   
white     1233.0  2561271.0     0.334     0.384  0.049        0.413   

          cuomo_gen  sliwa_gen  mamdani_vs_cuomo_gen  mamdani_h2h_prim  \
majority                                                                 
asian         0.424      0.104                 0.523             0.614   
black         0.329      0.026                 0.659             0.437   
hispanic      0.349      0.059                 0.626             0.553   
mixed         0.346      0.070                 0.625             0.646   
white         0.498      0.084         

## 3. Within-type: among EDs of the same demographic type, did the ones that swung to Trump go for Mamdani?

The between-type comparison mixes who lives there with how they moved. Within each majority group, regress Mamdani's 2025 measures on the 2020→2024 swing, holding the 2020 Trump level, income, education, and composition fixed. A positive coefficient on the swing means the EDs that moved right in 2024 were *more* Mamdani in 2025 than otherwise-similar EDs that did not; negative means the 2024 movers stayed away from him.

In [4]:
rows = []
for grp, s in list(e.groupby('majority')) + [('all', e)]:
    for dep in ['mamdani_vs_cuomo_gen', 'mamdani_h2h_prim', 'sliwa_gen_share', 'turnout_ratio_25_24']:
        w = s.pres24_2p if 'gen' in dep else s.prim_h2h_total
        sub = s.dropna(subset=[dep, 'swing_to_trump', 'trump_share_20', 'med_income', 'ba_share']); w = w.loc[sub.index]
        X = sm.add_constant(pd.DataFrame({'swing': sub.swing_to_trump, 'trump_20': sub.trump_share_20, 'log_income': np.log(sub.med_income), 'ba_share': sub.ba_share, 'hispanic': sub.hispanic_share, 'black': sub.black_nh_share, 'asian': sub.asian_nh_share}))
        r = sm.WLS(sub[dep], X, weights=w).fit(cov_type='HC1')
        rows.append({'group': grp, 'dep': dep, 'coef_swing': r.params['swing'], 'se_swing': r.bse['swing'], 'coef_trump20': r.params['trump_20'], 'se_trump20': r.bse['trump_20'], 'N': len(sub), 'R2': r.rsquared})
wt = pd.DataFrame(rows).set_index(['group', 'dep']).round(3)
print('Coefficients on the 2020->2024 swing and on the 2020 Trump level (0->1 changes), within group, controlling for log income, BA share, composition:'); print(wt)
print('Read: a swing coefficient of -0.5 means an ED that moved 10 points to Trump in 2024 was 5 points less Mamdani in 2025 than a similar ED that did not move.')
wt.to_csv(TABLES_DIR / 'nyc_within_group_models.csv')

Coefficients on the 2020->2024 swing and on the 2020 Trump level (0->1 changes), within group, controlling for log income, BA share, composition:
                               coef_swing  se_swing  coef_trump20  se_trump20  \
group    dep                                                                    
asian    mamdani_vs_cuomo_gen       0.034     0.103        -1.483       0.088   
         mamdani_h2h_prim           0.262     0.161        -1.419       0.098   
         sliwa_gen_share            0.032     0.041         0.320       0.037   
         turnout_ratio_25_24        0.620     0.180        -0.836       0.118   
black    mamdani_vs_cuomo_gen      -0.624     0.212        -1.650       0.123   
         mamdani_h2h_prim          -0.213     0.137        -2.328       0.201   
         sliwa_gen_share            0.022     0.021         0.319       0.043   
         turnout_ratio_25_24        7.730     2.268        -1.813       0.571   
hispanic mamdani_vs_cuomo_gen      -0.317   

## 4. Turnout: who showed up in 2025

Mamdani's primary and general were both higher-turnout than recent mayoral races. The ratio of 2025 general votes to 2024 presidential votes, and of primary ballots to 2024 votes, by demographic type and by 2024 Trump decile, says whether the 2025 electorate was the 2024 electorate or a subset of it.

In [5]:
print('Ratio of 2025 mayoral general votes to 2024 presidential votes, by majority group:'); print(by_maj[['gen25_per_24_vote', 'prim_per_24_vote']].round(3))
print('\nBy 2024 Trump decile:'); print(dec[['gen25_per_24_vote', 'prim_ballots_per_24_vote']].round(3))
# Where did Mamdani's general-election votes come from vs Harris's 2024 votes? share of each candidate's citywide total by majority group
src = e.groupby('majority').apply(lambda s: pd.Series({'harris_24_votes': s.pres24_harris.sum()/e.pres24_harris.sum(), 'trump_24_votes': s.pres24_trump.sum()/e.pres24_trump.sum(),
    'mamdani_gen_votes': s.gen25_mamdani.sum()/e.gen25_mamdani.sum(), 'cuomo_gen_votes': s.gen25_cuomo.sum()/e.gen25_cuomo.sum(), 'sliwa_gen_votes': s.gen25_sliwa.sum()/e.gen25_sliwa.sum(),
    'mamdani_prim_first': s.prim25_first_mamdani.sum()/e.prim25_first_mamdani.sum(), 'cuomo_prim_first': s.prim25_first_cuomo.sum()/e.prim25_first_cuomo.sum()}))
print('\nComposition of each candidate\'s citywide vote by ED majority group:'); print(src.round(3))
src.to_csv(TABLES_DIR / 'nyc_vote_composition_by_majority.csv')

Ratio of 2025 mayoral general votes to 2024 presidential votes, by majority group:
          gen25_per_24_vote  prim_per_24_vote
majority                                     
asian                 0.785             0.302
black                 0.668             0.384
hispanic              0.694             0.346
mixed                 0.814             0.401
white                 0.917             0.434

By 2024 Trump decile:
              gen25_per_24_vote  prim_ballots_per_24_vote
trump_decile                                             
1                         0.824                     0.559
2                         0.772                     0.480
3                         0.816                     0.491
4                         0.816                     0.456
5                         0.803                     0.421
6                         0.759                     0.378
7                         0.761                     0.355
8                         0.781                   

## 4b. Mamdani against the 2024 Democratic baseline

The cleanest ED-level comparison is Mamdani's November share against Harris's: both are general-election shares of the whole electorate in the same precinct a year apart. Where Mamdani ran furthest behind Harris, Democrats defected (to Cuomo) or stayed home; where he matched her, the 2024 Democratic vote held and any 2024 Trump vote went to Cuomo or Sliwa.

In [6]:
e['mamdani_minus_harris'] = e.mamdani_gen_share - e.harris_share
e['cuomo_sliwa_minus_trump'] = (e.gen25_cuomo + e.gen25_sliwa) / e.gen25_total - e.trump_share
t = e.groupby('swing_decile').apply(lambda s: pd.Series({'harris_24': s.pres24_harris.sum()/s.pres24_2p.sum(), 'mamdani_25': s.gen25_mamdani.sum()/s.gen25_total.sum(),
    'mamdani_minus_harris': s.gen25_mamdani.sum()/s.gen25_total.sum() - s.pres24_harris.sum()/s.pres24_2p.sum(),
    'trump_24': s.pres24_trump.sum()/s.pres24_2p.sum(), 'cuomo_plus_sliwa_25': (s.gen25_cuomo.sum() + s.gen25_sliwa.sum())/s.gen25_total.sum()}))
print('By swing decile: Mamdani 2025 vs Harris 2024, Cuomo+Sliwa 2025 vs Trump 2024 (shares of each contest vote):'); print(t.round(3))
t.to_csv(TABLES_DIR / 'nyc_mamdani_vs_harris_by_swing_decile.csv')
tg = e.groupby('majority').apply(lambda s: pd.Series({'harris_24': s.pres24_harris.sum()/s.pres24_2p.sum(), 'mamdani_25': s.gen25_mamdani.sum()/s.gen25_total.sum(),
    'mamdani_minus_harris': s.gen25_mamdani.sum()/s.gen25_total.sum() - s.pres24_harris.sum()/s.pres24_2p.sum(), 'trump_24': s.pres24_trump.sum()/s.pres24_2p.sum(), 'cuomo_plus_sliwa_25': (s.gen25_cuomo.sum() + s.gen25_sliwa.sum())/s.gen25_total.sum()}))
print('\nBy majority group:'); print(tg.round(3))
rows = []
for grp, s in list(e.groupby('majority')) + [('all', e)]:
    sub = s.dropna(subset=['mamdani_minus_harris', 'swing_to_trump', 'trump_share_20', 'med_income', 'ba_share'])
    X = sm.add_constant(pd.DataFrame({'swing': sub.swing_to_trump, 'trump_20': sub.trump_share_20, 'log_income': np.log(sub.med_income), 'ba_share': sub.ba_share, 'hispanic': sub.hispanic_share, 'black': sub.black_nh_share, 'asian': sub.asian_nh_share}))
    r = sm.WLS(sub.mamdani_minus_harris, X, weights=sub.pres24_2p).fit(cov_type='HC1')
    rows.append({'group': grp, 'coef_swing': r.params['swing'], 'se_swing': r.bse['swing'], 'coef_trump20': r.params['trump_20'], 'coef_ba': r.params['ba_share'], 'coef_log_income': r.params['log_income'], 'N': len(sub), 'R2': r.rsquared})
mh = pd.DataFrame(rows).set_index('group').round(3); print('\n(Mamdani 2025 - Harris 2024) on the swing, within group, with controls:'); print(mh)
mh.to_csv(TABLES_DIR / 'nyc_mamdani_minus_harris_models.csv')

By swing decile: Mamdani 2025 vs Harris 2024, Cuomo+Sliwa 2025 vs Trump 2024 (shares of each contest vote):
              harris_24  mamdani_25  mamdani_minus_harris  trump_24  \
swing_decile                                                          
1                 0.840       0.638                -0.202     0.160   
2                 0.798       0.563                -0.235     0.202   
3                 0.774       0.526                -0.247     0.226   
4                 0.708       0.479                -0.228     0.292   
5                 0.651       0.454                -0.196     0.349   
6                 0.643       0.444                -0.199     0.357   
7                 0.616       0.440                -0.176     0.384   
8                 0.616       0.480                -0.136     0.384   
9                 0.609       0.500                -0.109     0.391   
10                0.539       0.535                -0.004     0.461   

              cuomo_plus_sliwa_25  
swi

## 5. Figure

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, dep, lab in [(axes[0], 'mamdani_vs_cuomo_gen', 'Mamdani share of Mamdani+Cuomo, Nov 2025'), (axes[1], 'mamdani_h2h_prim', 'Mamdani vs Cuomo head-to-head, June 2025 primary')]:
    for grp, col in [('hispanic', 'tab:orange'), ('black', 'tab:purple'), ('white', 'tab:blue'), ('asian', 'tab:green'), ('mixed', 'grey')]:
        s = e[e.majority == grp]
        ax.scatter(s.swing_to_trump, s[dep], s=np.sqrt(s.pres24_2p)/2, alpha=0.35, c=col, label=f'{grp} majority (N={len(s)})', edgecolors='none')
    ax.set_xlabel('Swing to Trump, 2020 -> 2024 (two-party share points)'); ax.set_ylabel(lab); ax.set_xlim(-0.15, 0.45); ax.set_ylim(0, 1); ax.grid(alpha=0.3); ax.axvline(0, color='k', lw=0.5)
axes[0].legend(fontsize=8, loc='upper right')
fig.suptitle('NYC election districts: 2020->2024 swing to Trump vs 2025 Mamdani vote')
plt.tight_layout(); plt.savefig(FIGURES_DIR / 'nyc_trump24_vs_mamdani25.png', dpi=150); print('saved figure')

saved figure


## Limitations

- **Ecological.** ED-level co-movement does not identify individual voters. A Hispanic ED that went 40% Trump in 2024 and 60% Mamdani in 2025 could be the same voters or different subsets of the same neighborhood.
- **The 2020 baseline is crosswalked** from the pre-2022 ED lines through 2020 census blocks, assuming votes are distributed within a 2020 ED in proportion to block population; 98% of 2020 votes are carried over. Swings in small or oddly shaped EDs carry crosswalk noise.
- **Electorates differ.** Primary: registered Democrats, ranked-choice. 2025 general: three-way, off-year, ~a third smaller than presidential. Turnout ratios are reported so the reader can see which neighborhoods dropped out.
- **ED boundary drift** across nyed_24d / 25b / 25d; matched on AD-ED number; EDs "combined into" another are dropped.
- **Demographics are 2020 Census population, not 2024–25 electorate**; block-group ACS income and education are 2019–2023 averages.